# ⚙️ Predictive Maintenance — ETL Pipeline

Production-ready ETL pipeline for IoT sensor data preprocessing.

### Pipeline Steps
| # | Step | Description |
|---|------|-------------|
| 1 | Data Validation | Structure checks, missing column detection, duplicates |
| 2 | Missing Value Imputation | Median / mean fill, mode for categoricals |
| 3 | Outlier Detection & Treatment | IQR / Z-score clipping |
| 4 | Feature Engineering | 20+ physics & operational features |
| 5 | Feature Matrix Assembly | Numeric selection, NaN/inf cleanup |
| 6 | Train/Test Split | Stratified, leak-free |
| 7 | Feature Scaling | RobustScaler (fit on train only) |
| 8 | Feature Selection | Variance, correlation, mutual information |
| 9 | Class Imbalance | SMOTE on training data only |
| 10 | Artifact Persistence | Save scaler, bounds, config for inference |

In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Imports
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from scipy import stats
from imblearn.over_sampling import SMOTE
import joblib
import json
import logging
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ── Logger ───────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('ETL_Pipeline')

print("Imports loaded.")

Imports loaded.


## 0. Configuration & Pipeline State
All tuneable parameters and every piece of state that the pipeline carries between steps lives here.

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Pipeline Configuration
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

config = {
    'test_size'              : 0.2,
    'random_state'           : 42,
    'missing_strategy'       : 'median',       # 'median', 'mean', 'knn'
    'outlier_method'         : 'iqr',          # 'iqr', 'zscore'
    'outlier_threshold'      : 1.5,            # IQR multiplier
    'zscore_threshold'       : 3.5,            # Z-score threshold
    'scaling_method'         : 'robust',       # 'robust', 'standard'
    'apply_smote'            : True,
    'smote_ratio'            : 0.5,            # Minority / Majority ratio
    'feature_selection'      : True,
    'mi_threshold'           : 0.01,           # Mutual info threshold
    'correlation_threshold'  : 0.95,           # Remove highly correlated pairs
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Pipeline State (populated as we go)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

scaler                        = None
feature_names                 = []
feature_names_before_selection = []
imputation_values             = {}
outlier_bounds                = {}
selected_features             = []
type_mapping                  = {'L': 0, 'M': 1, 'H': 2}
shift_mapping                 = {'Day': 0, 'Evening': 1, 'Night': 2}
is_fitted                     = False

pipeline_metadata = {
    'steps_completed'            : [],
    'timestamp'                  : None,
    'original_shape'             : None,
    'final_shape'                : None,
    'features_created'           : [],
    'features_removed'           : [],
    'missing_values_filled'      : 0,
    'outliers_treated'           : 0,
    'class_distribution_before'  : {},
    'class_distribution_after'   : {},
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Column Groups
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

sensor_cols = [
    'Air_Temp_K', 'Process_Temp_K', 'Rot_Speed_RPM',
    'Torque_Nm', 'Tool_Wear_Min', 'Vibration_mm_s', 'Pressure_bar'
]
categorical_cols   = ['Type', 'Shift']
id_cols            = ['Timestamp', 'Machine_ID']
target_col         = 'Machine_Failure'
failure_mode_cols  = [
    'Failure_TWF', 'Failure_HDF', 'Failure_PWF',
    'Failure_OSF', 'Failure_RNF', 'Failure_Mode'
]

print("Configuration & state initialised.")
for k, v in config.items():
    print(f"   {k:.<35s} {v}")

Configuration & state initialised.
   test_size.......................... 0.2
   random_state....................... 42
   missing_strategy................... median
   outlier_method..................... iqr
   outlier_threshold.................. 1.5
   zscore_threshold................... 3.5
   scaling_method..................... robust
   apply_smote........................ True
   smote_ratio........................ 0.5
   feature_selection.................. True
   mi_threshold....................... 0.01
   correlation_threshold.............. 0.95


## 0b. Load Raw Data

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Load data
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

pipeline_metadata['timestamp'] = datetime.now().isoformat()

try:
    df_raw = pd.read_csv('sensor_data.csv')
    print(f"Loaded sensor_data.csv: {df_raw.shape}")
except FileNotFoundError:
    raise FileNotFoundError("sensor_data.csv not found. Run the data generator notebook first.")

df_raw.head()

Loaded sensor_data.csv: (50000, 19)


,Timestamp,Machine_ID,Type,Shift,Air_Temp_K,Process_Temp_K,Rot_Speed_RPM,Torque_Nm,Tool_Wear_Min,Vibration_mm_s,Pressure_bar,Machine_Age_Years,Failure_TWF,Failure_HDF,Failure_PWF,Failure_OSF,Failure_RNF,Machine_Failure,Failure_Mode
0,2024-01-01 00:00:00,MILL-001,L,Night,295.37,305.26,3133.3,51.48,3.5,0.702,5.61,0.9,0,0,0,0,0,0,NaN
1,2024-01-01 00:00:00,MILL-009,H,Night,294.97,304.90,2794.3,47.72,5.2,0.474,5.57,6.3,0,0,0,0,0,0,NaN
2,2024-01-01 00:00:00,MILL-008,L,Night,293.01,302.97,2761.5,38.77,2.0,0.548,5.85,6.7,0,0,0,0,0,0,NaN
3,2024-01-01 00:00:00,MILL-007,H,Night,296.62,307.28,2759.1,61.91,1.1,0.473,6.38,5.0,0,0,0,0,0,0,NaN
4,2024-01-01 00:00:00,MILL-006,M,Night,292.29,301.90,2829.1,28.36,2.6,0.718,5.83,2.8,0,0,0,0,0,0,NaN


## Step 1 — Data Validation
Checks:
- Required columns exist
- No fully empty columns
- Minimum row count
- Duplicate rows

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 1: Data Validation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 1: Data Validation")
print("=" * 50)

data = df_raw.copy()

validation_report = {
    'shape'           : data.shape,
    'columns_found'   : list(data.columns),
    'missing_columns' : [],
    'dtype_issues'    : [],
    'warnings'        : [],
    'passed'          : True,
}

# ── Required columns check ───────────────
minimum_required = [
    'Type', 'Air_Temp_K', 'Process_Temp_K',
    'Rot_Speed_RPM', 'Torque_Nm', 'Tool_Wear_Min'
]

for col in minimum_required:
    if col not in data.columns:
        validation_report['missing_columns'].append(col)
        validation_report['passed'] = False

if not validation_report['passed']:
    raise ValueError(
        f"Missing required columns: {validation_report['missing_columns']}. "
        f"Expected at minimum: {minimum_required}"
    )

# ── Fully empty columns ─────────────────
empty_cols = data.columns[data.isnull().all()].tolist()
if empty_cols:
    validation_report['warnings'].append(f"Fully empty columns: {empty_cols}")
    logger.warning(f"Fully empty columns found: {empty_cols}")

# ── Minimum rows ────────────────────────
if len(data) < 100:
    validation_report['warnings'].append("Very small dataset (<100 rows)")
    logger.warning("Dataset has fewer than 100 rows")

# ── Duplicates ──────────────────────────
n_duplicates = data.duplicated().sum()
if n_duplicates > 0:
    validation_report['warnings'].append(f"Found {n_duplicates} duplicate rows")
    logger.warning(f"Found {n_duplicates} duplicate rows")

# ── Summary ─────────────────────────────
print(f"  Shape: {data.shape}")
print(f"  Columns: {len(data.columns)}")
print(f"  Missing values: {data.isnull().sum().sum():,}")
print(f"  Duplicates: {n_duplicates}")
print(f"  Validation: {'PASSED ' if validation_report['passed'] else 'FAILED'}")

pipeline_metadata['original_shape'] = data.shape
pipeline_metadata['steps_completed'].append('validation')

STEP 1: Data Validation
  Shape: (50000, 19)
  Columns: 19
  Missing values: 55,322
  Duplicates: 0
  Validation: PASSED 


## Step 2 — Missing Value Imputation
- **Numerical sensors:** Median imputation (robust to outliers)
- **Categorical:** Mode imputation
- Imputation values are **saved** for later use in inference

In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 2: Missing Value Imputation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 2: Missing Value Imputation")
print("=" * 50)

total_missing_before = data.isnull().sum().sum()

available_sensors     = [c for c in sensor_cols if c in data.columns]
available_categorical = [c for c in categorical_cols if c in data.columns]

# ── Fit imputation values (training mode) ─
imputation_values = {}

for col in available_sensors:
    if data[col].isnull().any():
        if config['missing_strategy'] == 'median':
            imputation_values[col] = data[col].median()
        elif config['missing_strategy'] == 'mean':
            imputation_values[col] = data[col].mean()
        else:
            imputation_values[col] = data[col].median()

        n_missing = data[col].isnull().sum()
        print(
            f"  {col}: {n_missing} missing → "
            f"fill with {config['missing_strategy']} ({imputation_values[col]:.2f})"
        )

for col in available_categorical:
    if data[col].isnull().any():
        imputation_values[col] = data[col].mode()[0]
        print(
            f"  {col}: {data[col].isnull().sum()} missing → "
            f"fill with mode ({imputation_values[col]})"
        )

# ── Apply imputation ─────────────────────
for col, fill_value in imputation_values.items():
    if col in data.columns:
        data[col].fillna(fill_value, inplace=True)

# ── Drop rows where target is missing ────
if target_col in data.columns:
    before_len = len(data)
    data = data.dropna(subset=[target_col])
    dropped = before_len - len(data)
    if dropped > 0:
        print(f"  Dropped {dropped} rows with missing target")

total_missing_after = data.isnull().sum().sum()
filled = total_missing_before - total_missing_after

pipeline_metadata['missing_values_filled'] = filled
print(f"  Total filled: {filled:,} values")
print(f"  Remaining missing: {total_missing_after:,}")
pipeline_metadata['steps_completed'].append('imputation')

STEP 2: Missing Value Imputation
  Air_Temp_K: 1156 missing → fill with median (294.48)
  Process_Temp_K: 653 missing → fill with median (304.64)
  Rot_Speed_RPM: 703 missing → fill with median (2854.10)
  Torque_Nm: 1269 missing → fill with median (39.26)
  Tool_Wear_Min: 640 missing → fill with median (92.60)
  Vibration_mm_s: 1449 missing → fill with median (1.70)
  Pressure_bar: 1080 missing → fill with median (6.28)
  Total filled: 6,950 values
  Remaining missing: 48,372


## Step 3 — Outlier Detection & Treatment
- **IQR method:** values outside `[Q1 − k·IQR, Q3 + k·IQR]` are clipped
- **Z-score method:** values with `|z| > threshold` are clipped
- Bounds are **saved** for inference reuse

In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 3: Outlier Detection & Treatment
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 3: Outlier Detection & Treatment")
print("=" * 50)

available_sensors = [c for c in sensor_cols if c in data.columns]
total_outliers    = 0
outlier_bounds    = {}

for col in available_sensors:
    col_data = data[col].dropna()

    if config['outlier_method'] == 'iqr':
        Q1    = col_data.quantile(0.25)
        Q3    = col_data.quantile(0.75)
        IQR   = Q3 - Q1
        k     = config['outlier_threshold']
        lower = Q1 - k * IQR
        upper = Q3 + k * IQR

    elif config['outlier_method'] == 'zscore':
        mean  = col_data.mean()
        std   = col_data.std()
        t     = config['zscore_threshold']
        lower = mean - t * std
        upper = mean + t * std

    else:
        lower = col_data.min()
        upper = col_data.max()

    outlier_bounds[col] = {'lower': lower, 'upper': upper}

# ── Apply clipping ───────────────────────
for col in available_sensors:
    if col in outlier_bounds:
        bounds     = outlier_bounds[col]
        n_outliers = ((data[col] < bounds['lower']) | (data[col] > bounds['upper'])).sum()
        total_outliers += n_outliers

        data[col] = data[col].clip(lower=bounds['lower'], upper=bounds['upper'])

        if n_outliers > 0:
            print(
                f"  {col}: {n_outliers} outliers clipped "
                f"[{bounds['lower']:.2f}, {bounds['upper']:.2f}]"
            )

pipeline_metadata['outliers_treated'] = total_outliers
print(f"  Total outliers treated: {total_outliers:,}")
pipeline_metadata['steps_completed'].append('outlier_treatment')

STEP 3: Outlier Detection & Treatment
  Air_Temp_K: 424 outliers clipped [288.33, 300.61]
  Process_Temp_K: 499 outliers clipped [298.00, 311.24]
  Rot_Speed_RPM: 66 outliers clipped [2476.65, 3220.25]
  Torque_Nm: 385 outliers clipped [15.89, 62.48]
  Vibration_mm_s: 116 outliers clipped [-0.57, 4.07]
  Pressure_bar: 622 outliers clipped [5.15, 7.39]
  Total outliers treated: 2,112


## Step 4 — Feature Engineering
Creates **20+ domain-specific features** across 8 categories:

| Category | Examples |
|----------|---------|
| Physical Indicators | Power, Strain, Efficiency Proxy |
| Temperature | Temp Diff, Ratio, Heat Risk, Overheat Flag |
| Vibration | Vib-Wear Interaction, Risk Level, Vib-Torque |
| Pressure | Deviation, Pressure-Torque, Low Pressure Flag |
| Operational | Wear Stage, Load Regime, Wear Rate, Age interactions |
| Statistical | Z-scores per sensor, Composite Anomaly Score |
| Time-based | Hour (sin/cos), Day of Week, Is Weekend |
| Encoding | Type → ordinal, Shift → ordinal |

In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 4: Feature Engineering
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 4: Feature Engineering")
print("=" * 50)

new_features = []

# ══════════════════════════════════════════
# 4.1 PHYSICAL INDICATORS
# ══════════════════════════════════════════

# Power (Watts) = Torque × Angular Velocity
data['Power_W'] = data['Torque_Nm'] * data['Rot_Speed_RPM'] * (2 * np.pi / 60)
new_features.append('Power_W')

# Strain Level = Torque × Tool Wear (overstrain risk)
data['Strain_Level'] = data['Torque_Nm'] * data['Tool_Wear_Min']
new_features.append('Strain_Level')

# Torque-Speed Ratio (efficiency indicator)
data['Torque_Speed_Ratio'] = data['Torque_Nm'] / (data['Rot_Speed_RPM'] + 1)
new_features.append('Torque_Speed_Ratio')

# Mechanical Efficiency Proxy
data['Efficiency_Proxy'] = data['Power_W'] / (data['Tool_Wear_Min'] + 1)
new_features.append('Efficiency_Proxy')

# ══════════════════════════════════════════
# 4.2 TEMPERATURE FEATURES
# ══════════════════════════════════════════

# Temperature Difference (heat dissipation indicator)
data['Temp_Diff'] = data['Process_Temp_K'] - data['Air_Temp_K']
new_features.append('Temp_Diff')

# Temperature Ratio
data['Temp_Ratio'] = data['Process_Temp_K'] / (data['Air_Temp_K'] + 1)
new_features.append('Temp_Ratio')

# Heat Dissipation Risk (temp_diff × speed interaction)
data['Heat_Risk'] = data['Temp_Diff'] * data['Rot_Speed_RPM'] / 1000
new_features.append('Heat_Risk')

# Overheating Flag (process temp in danger zone)
data['Overheat_Flag'] = (data['Process_Temp_K'] > 312).astype(int)
new_features.append('Overheat_Flag')

# ══════════════════════════════════════════
# 4.3 VIBRATION FEATURES
# ══════════════════════════════════════════

if 'Vibration_mm_s' in data.columns:
    # Vibration-Wear Interaction (worn tool + high vibration = danger)
    data['Vib_Wear_Interaction'] = data['Vibration_mm_s'] * data['Tool_Wear_Min']
    new_features.append('Vib_Wear_Interaction')

    # Vibration Risk Level (categorical bins)
    data['Vib_Risk_Level'] = pd.cut(
        data['Vibration_mm_s'],
        bins=[0, 1.0, 2.0, 3.5, float('inf')],
        labels=[0, 1, 2, 3]   # Safe, Warning, High, Critical
    ).astype(float).fillna(0).astype(int)
    new_features.append('Vib_Risk_Level')

    # Vibration-Torque Interaction
    data['Vib_Torque'] = data['Vibration_mm_s'] * data['Torque_Nm']
    new_features.append('Vib_Torque')

# ══════════════════════════════════════════
# 4.4 PRESSURE FEATURES
# ══════════════════════════════════════════

if 'Pressure_bar' in data.columns:
    # Pressure deviation from nominal (6 bar)
    data['Pressure_Deviation'] = abs(data['Pressure_bar'] - 6.0)
    new_features.append('Pressure_Deviation')

    # Pressure-Torque Interaction
    data['Pressure_Torque'] = data['Pressure_bar'] * data['Torque_Nm']
    new_features.append('Pressure_Torque')

    # Low Pressure Flag (potential leak)
    data['Low_Pressure_Flag'] = (data['Pressure_bar'] < 4.5).astype(int)
    new_features.append('Low_Pressure_Flag')

# ══════════════════════════════════════════
# 4.5 OPERATIONAL FEATURES
# ══════════════════════════════════════════

# Tool Wear Stage (lifecycle binning)
data['Wear_Stage'] = pd.cut(
    data['Tool_Wear_Min'],
    bins=[-1, 50, 120, 200, float('inf')],
    labels=[0, 1, 2, 3]   # New, Mid-Life, Worn, Critical
).astype(float).fillna(0).astype(int)
new_features.append('Wear_Stage')

# Operating Load Regime
data['Load_Regime'] = pd.cut(
    data['Power_W'],
    bins=[-float('inf'), 3000, 6000, 9000, float('inf')],
    labels=[0, 1, 2, 3]   # Low, Normal, High, Extreme
).astype(float).fillna(0).astype(int)
new_features.append('Load_Regime')

# Wear Rate Approximation (wear per unit torque)
data['Wear_Rate'] = data['Tool_Wear_Min'] / (data['Torque_Nm'] + 1)
new_features.append('Wear_Rate')

# Machine Age Interactions
if 'Machine_Age_Years' in data.columns:
    data['Age_Stress'] = data['Machine_Age_Years'] * data['Strain_Level'] / 100
    new_features.append('Age_Stress')

    data['Age_Vibration'] = data['Machine_Age_Years'] * data.get('Vibration_mm_s', 0)
    new_features.append('Age_Vibration')

# ══════════════════════════════════════════
# 4.6 STATISTICAL FEATURES
# ══════════════════════════════════════════

# Z-scores for key sensors (how unusual is this reading?)
zscore_cols = ['Rot_Speed_RPM', 'Torque_Nm', 'Tool_Wear_Min']
if 'Vibration_mm_s' in data.columns:
    zscore_cols.append('Vibration_mm_s')

for col in zscore_cols:
    col_mean = data[col].mean()
    col_std  = data[col].std()
    if col_std > 0:
        data[f'{col}_zscore'] = abs((data[col] - col_mean) / col_std)
        new_features.append(f'{col}_zscore')

# Composite Anomaly Score (average of z-scores)
zscore_features = [f for f in new_features if '_zscore' in f]
if zscore_features:
    data['Anomaly_Score'] = data[zscore_features].mean(axis=1)
    new_features.append('Anomaly_Score')

# ══════════════════════════════════════════
# 4.7 TIME-BASED FEATURES
# ══════════════════════════════════════════

if 'Timestamp' in data.columns:
    data['Timestamp'] = pd.to_datetime(data['Timestamp'])

    data['Hour'] = data['Timestamp'].dt.hour
    new_features.append('Hour')

    # Cyclical encoding (sine/cosine) for hour
    data['Hour_Sin'] = np.sin(2 * np.pi * data['Hour'] / 24)
    data['Hour_Cos'] = np.cos(2 * np.pi * data['Hour'] / 24)
    new_features.extend(['Hour_Sin', 'Hour_Cos'])

    data['DayOfWeek'] = data['Timestamp'].dt.dayofweek
    new_features.append('DayOfWeek')

    data['Is_Weekend'] = (data['DayOfWeek'] >= 5).astype(int)
    new_features.append('Is_Weekend')

# ══════════════════════════════════════════
# 4.8 ENCODE CATEGORICALS
# ══════════════════════════════════════════

if 'Type' in data.columns:
    data['Type_Encoded'] = data['Type'].map(type_mapping).fillna(0).astype(int)
    new_features.append('Type_Encoded')

if 'Shift' in data.columns:
    data['Shift_Encoded'] = data['Shift'].map(shift_mapping).fillna(0).astype(int)
    new_features.append('Shift_Encoded')

# ── Summary ──────────────────────────────
print(f"  Created {len(new_features)} new features:")
for i, feat in enumerate(new_features, 1):
    print(f"    {i:2d}. {feat}")

pipeline_metadata['features_created'] = new_features
pipeline_metadata['steps_completed'].append('feature_engineering')

STEP 4: Feature Engineering
  Created 31 new features:
     1. Power_W
     2. Strain_Level
     3. Torque_Speed_Ratio
     4. Efficiency_Proxy
     5. Temp_Diff
     6. Temp_Ratio
     7. Heat_Risk
     8. Overheat_Flag
     9. Vib_Wear_Interaction
    10. Vib_Risk_Level
    11. Vib_Torque
    12. Pressure_Deviation
    13. Pressure_Torque
    14. Low_Pressure_Flag
    15. Wear_Stage
    16. Load_Regime
    17. Wear_Rate
    18. Age_Stress
    19. Age_Vibration
    20. Rot_Speed_RPM_zscore
    21. Torque_Nm_zscore
    22. Tool_Wear_Min_zscore
    23. Vibration_mm_s_zscore
    24. Anomaly_Score
    25. Hour
    26. Hour_Sin
    27. Hour_Cos
    28. DayOfWeek
    29. Is_Weekend
    30. Type_Encoded
    31. Shift_Encoded


## Step 5 — Build Feature Matrix
Select only numeric columns suitable for modelling — excluding IDs, timestamps, target, and failure-mode labels.

In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 5: Build Feature Matrix
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 5: Building Feature Matrix")
print("=" * 50)

# Columns to exclude
exclude_cols = set(
    id_cols +
    categorical_cols +
    failure_mode_cols +
    [target_col, 'Machine_Age_Years', 'Hour']
)

# All numeric columns minus exclusions
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

X = data[feature_cols].copy()

# Handle remaining NaN
if X.isnull().any().any():
    logger.warning(f"  Found {X.isnull().sum().sum()} remaining NaN values — filling with 0")
    X = X.fillna(0)

# Replace infinities
X = X.replace([np.inf, -np.inf], 0)

# Extract target
y = data[target_col] if target_col in data.columns else None

feature_names_before_selection = feature_cols

print(f"  Feature matrix shape: {X.shape}")
print(f"  Feature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    print(f"    {i:2d}. {col}")

if y is not None:
    print(f"  Target distribution:")
    print(f"    Normal (0): {(y == 0).sum():,}")
    print(f"    Failure (1): {(y == 1).sum():,}")
    print(f"    Failure rate: {y.mean():.2%}")

pipeline_metadata['steps_completed'].append('build_matrix')

STEP 5: Building Feature Matrix
  Feature matrix shape: (50000, 37)
  Feature columns (37):
     1. Air_Temp_K
     2. Process_Temp_K
     3. Rot_Speed_RPM
     4. Torque_Nm
     5. Tool_Wear_Min
     6. Vibration_mm_s
     7. Pressure_bar
     8. Power_W
     9. Strain_Level
    10. Torque_Speed_Ratio
    11. Efficiency_Proxy
    12. Temp_Diff
    13. Temp_Ratio
    14. Heat_Risk
    15. Overheat_Flag
    16. Vib_Wear_Interaction
    17. Vib_Risk_Level
    18. Vib_Torque
    19. Pressure_Deviation
    20. Pressure_Torque
    21. Low_Pressure_Flag
    22. Wear_Stage
    23. Load_Regime
    24. Wear_Rate
    25. Age_Stress
    26. Age_Vibration
    27. Rot_Speed_RPM_zscore
    28. Torque_Nm_zscore
    29. Tool_Wear_Min_zscore
    30. Vibration_mm_s_zscore
    31. Anomaly_Score
    32. Hour_Sin
    33. Hour_Cos
    34. DayOfWeek
    35. Is_Weekend
    36. Type_Encoded
    37. Shift_Encoded
  Target distribution:
    Normal (0): 48,372
    Failure (1): 1,628
    Failure rate: 3.26%


## Step 6 — Train / Test Split
Stratified split happens **before** scaling to prevent data leakage.

In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 6: Train/Test Split
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 6: Train/Test Split")
print("=" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = config['test_size'],
    random_state = config['random_state'],
    stratify     = y,
)

print(f"  Training set: {X_train.shape}")
print(f"    Failure rate: {y_train.mean():.2%}")
print(f"  Test set: {X_test.shape}")
print(f"    Failure rate: {y_test.mean():.2%}")

pipeline_metadata['class_distribution_before'] = {
    'train_0': int((y_train == 0).sum()),
    'train_1': int((y_train == 1).sum()),
    'test_0' : int((y_test == 0).sum()),
    'test_1' : int((y_test == 1).sum()),
}
pipeline_metadata['steps_completed'].append('split')

STEP 6: Train/Test Split
  Training set: (40000, 37)
    Failure rate: 3.26%
  Test set: (10000, 37)
    Failure rate: 3.26%


## Step 7 — Feature Scaling
`RobustScaler` (resistant to outliers) is **fit on training data only**, then applied to both train and test.

In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 7: Feature Scaling
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 7: Feature Scaling")
print("=" * 50)

if config['scaling_method'] == 'robust':
    scaler = RobustScaler()
else:
    scaler = StandardScaler()

# Fit on train, transform both
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

print(f"  Fitted {config['scaling_method']} scaler on training data")
print(f"  Train scaled shape: {X_train_scaled.shape}")
print(f"  Test  scaled shape: {X_test_scaled.shape}")

pipeline_metadata['steps_completed'].append('scaling')

STEP 7: Feature Scaling
  Fitted robust scaler on training data
  Train scaled shape: (40000, 37)
  Test  scaled shape: (10000, 37)


## Step 8 — Feature Selection
Three-stage filter:
1. **Remove near-zero variance** features (< 0.001)
2. **Remove highly correlated** pairs (> 0.95)
3. **Mutual Information** ranking — drop features below threshold

In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 8: Feature Selection (Improved)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 8: Feature Selection")
print("=" * 50)

if config['feature_selection']:

    selection_report = {
        'initial_features'         : len(X_train_scaled.columns),
        'removed_low_variance'     : [],
        'removed_high_correlation' : [],
        'mutual_info_scores'       : {},
        'final_features'           : 0,
    }

    features_to_keep = list(X_train_scaled.columns)

    # ── 8.1 Near-zero variance ───────────────
    variances = X_train_scaled.var()
    low_var   = variances[variances < 0.001].index.tolist()
    if low_var:
        features_to_keep = [f for f in features_to_keep if f not in low_var]
        selection_report['removed_low_variance'] = low_var
        print(f"  Removed {len(low_var)} low-variance features: {low_var}")

    # ── 8.2 High correlation ─────────────────
    if len(features_to_keep) > 1:
        corr_matrix    = X_train_scaled[features_to_keep].corr().abs()
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        threshold = config['correlation_threshold']
        high_corr = [
            col for col in upper_triangle.columns
            if any(upper_triangle[col] > threshold)
        ]
        if high_corr:
            features_to_keep = [f for f in features_to_keep if f not in high_corr]
            selection_report['removed_high_correlation'] = high_corr
            print(f"  Removed {len(high_corr)} highly correlated features (> {threshold}): {high_corr}")

    # ── 8.3 Mutual Information ───────────────
    if len(features_to_keep) > 0:
        mi_scores = mutual_info_classif(
            X_train_scaled[features_to_keep], y_train,
            random_state=config['random_state'],
            n_neighbors=5,
        )
        mi_series = pd.Series(mi_scores, index=features_to_keep).sort_values(ascending=False)
        selection_report['mutual_info_scores'] = mi_series.to_dict()

        # Remove features below MI threshold
        low_mi = mi_series[mi_series < config['mi_threshold']].index.tolist()
        filtered = [f for f in features_to_keep if f not in low_mi]

        # ───────────────────────────────────────────────
        #  NEW RULE: Keep at least 50% of original features
        # ───────────────────────────────────────────────
        min_features = max(1, selection_report['initial_features'] // 2)

        if len(filtered) < min_features:
            print(f"  MI threshold removed too many features — keeping top {min_features} instead.")
            filtered = mi_series.head(min_features).index.tolist()
        else:
            print(f"  Removed {len(low_mi)} low mutual-info features: {low_mi}")

        features_to_keep = filtered

        print("\n  Top 15 Features by Mutual Information:")
        for i, (feat, score) in enumerate(mi_series.head(15).items(), 1):
            print(f"    {i:2d}. {feat:30s} MI = {score:.4f}")

    selected_features = features_to_keep
    feature_names     = selected_features
    selection_report['final_features'] = len(features_to_keep)

    print(
        f"\n  Feature selection: {selection_report['initial_features']} → "
        f"{selection_report['final_features']} features"
    )

    # Apply selection
    X_train_scaled = X_train_scaled[selected_features]
    X_test_scaled  = X_test_scaled[selected_features]

else:
    feature_names = list(X_train_scaled.columns)
    print("  Feature selection disabled — keeping all features.")

pipeline_metadata['steps_completed'].append('feature_selection')

STEP 8: Feature Selection
  Removed 2 low-variance features: ['Overheat_Flag', 'Low_Pressure_Flag']
  Removed 6 highly correlated features (> 0.95): ['Power_W', 'Torque_Speed_Ratio', 'Temp_Ratio', 'Vib_Wear_Interaction', 'Pressure_Torque', 'Wear_Stage']
  MI threshold removed too many features — keeping top 18 instead.

  Top 15 Features by Mutual Information:
     1. Vib_Torque                     MI = 0.0047
     2. Strain_Level                   MI = 0.0043
     3. Tool_Wear_Min                  MI = 0.0041
     4. Vibration_mm_s                 MI = 0.0032
     5. Tool_Wear_Min_zscore           MI = 0.0025
     6. Vib_Risk_Level                 MI = 0.0024
     7. Age_Vibration                  MI = 0.0023
     8. Wear_Rate                      MI = 0.0022
     9. Age_Stress                     MI = 0.0021
    10. Efficiency_Proxy               MI = 0.0020
    11. Torque_Nm                      MI = 0.0019
    12. Vibration_mm_s_zscore          MI = 0.0012
    13. Temp_Diff        

## Step 9 — Class Imbalance Handling (SMOTE)
SMOTE is applied to **training data only** — the test set stays untouched to reflect real-world class distribution.

In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 9: Class Imbalance — SMOTE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 50)
print("STEP 9: Class Imbalance Handling (SMOTE)")
print("=" * 50)

print(f"  Before SMOTE:")
print(f"    Normal  (0): {(y_train == 0).sum():,}")
print(f"    Failure (1): {(y_train == 1).sum():,}")

# Keep unbalanced copies for reference
X_train_unbalanced = X_train_scaled.copy()
y_train_unbalanced = y_train.copy()

if config['apply_smote']:
    smote = SMOTE(
        sampling_strategy = config['smote_ratio'],
        random_state      = config['random_state'],
        k_neighbors       = 5,
    )

    X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

    # Convert back to DataFrame / Series with feature names
    X_train_balanced = pd.DataFrame(X_resampled, columns=X_train_scaled.columns)
    y_train_balanced = pd.Series(y_resampled, name=y_train.name)

    print(f"  After SMOTE:")
    print(f"    Normal  (0): {(y_train_balanced == 0).sum():,}")
    print(f"    Failure (1): {(y_train_balanced == 1).sum():,}")

    pipeline_metadata['class_distribution_after'] = {
        'train_0': int((y_train_balanced == 0).sum()),
        'train_1': int((y_train_balanced == 1).sum()),
    }
    pipeline_metadata['steps_completed'].append('imbalance_smote')

else:
    X_train_balanced = X_train_scaled.copy()
    y_train_balanced = y_train.copy()
    print("  SMOTE disabled in config — skipping")
    pipeline_metadata['steps_completed'].append('imbalance_skipped')

STEP 9: Class Imbalance Handling (SMOTE)
  Before SMOTE:
    Normal  (0): 38,698
    Failure (1): 1,302
  After SMOTE:
    Normal  (0): 38,698
    Failure (1): 19,349


## Pipeline Complete — Final Summary

In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Mark fitted & final metadata
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

is_fitted = True

pipeline_metadata['final_shape'] = {
    'X_train': X_train_balanced.shape,
    'X_test' : X_test_scaled.shape,
}

print("\n" + "=" * 60)
print("  TRAINING ETL PIPELINE COMPLETE")
print("=" * 60)
print(f"  Final training features : {X_train_balanced.shape[1]}")
print(f"  Training samples        : {len(X_train_balanced):,}")
print(f"  Test samples            : {len(X_test_scaled):,}")

# Bundle results
results = {
    'X_train'             : X_train_balanced,
    'X_test'              : X_test_scaled,
    'y_train'             : y_train_balanced,
    'y_test'              : y_test,
    'X_train_unbalanced'  : X_train_unbalanced,
    'y_train_unbalanced'  : y_train_unbalanced,
    'feature_names'       : feature_names,
    'metadata'            : pipeline_metadata,
    'raw_engineered_data' : data,
}

print("\nPIPELINE RESULTS:")
print(f"  X_train shape : {results['X_train'].shape}")
print(f"  X_test shape  : {results['X_test'].shape}")
print(f"  Features used : {len(results['feature_names'])}")
print(f"  Feature names : {results['feature_names']}")


  TRAINING ETL PIPELINE COMPLETE
  Final training features : 18
  Training samples        : 58,047
  Test samples            : 10,000

PIPELINE RESULTS:
  X_train shape : (58047, 18)
  X_test shape  : (10000, 18)
  Features used : 18
  Feature names : ['Vib_Torque', 'Strain_Level', 'Tool_Wear_Min', 'Vibration_mm_s', 'Tool_Wear_Min_zscore', 'Vib_Risk_Level', 'Age_Vibration', 'Wear_Rate', 'Age_Stress', 'Efficiency_Proxy', 'Torque_Nm', 'Vibration_mm_s_zscore', 'Temp_Diff', 'Type_Encoded', 'Heat_Risk', 'Shift_Encoded', 'DayOfWeek', 'Process_Temp_K']


## Step 10 — Save Artifacts
Persist everything needed to run the inference pipeline later:
- `scaler.pkl` — fitted scaler
- `etl_config.json` — pipeline configuration
- `feature_names.json` — selected feature names
- `imputation_values.json` — fill values
- `outlier_bounds.json` — clipping bounds
- `pipeline_metadata.json` — full run metadata

In [15]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Save Artifacts
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

artifact_dir = 'artifacts'
os.makedirs(artifact_dir, exist_ok=True)

# ── Scaler ───────────────────────────────
joblib.dump(scaler, os.path.join(artifact_dir, 'scaler.pkl'))

# ── JSON artifacts ───────────────────────
json_artifacts = {
    'etl_config.json'        : config,
    'feature_names.json'     : feature_names,
    'feature_names_all.json' : feature_names_before_selection,
    'imputation_values.json' : {
        k: float(v) if isinstance(v, (np.floating, float)) else v
        for k, v in imputation_values.items()
    },
    'outlier_bounds.json'    : {
        k: {kk: float(vv) for kk, vv in v.items()}
        for k, v in outlier_bounds.items()
    },
    'type_mapping.json'      : type_mapping,
    'shift_mapping.json'     : shift_mapping,
}

# Safely serialise metadata
metadata_safe = {}
for key, value in pipeline_metadata.items():
    if isinstance(value, dict):
        metadata_safe[key] = {
            str(k): (list(v) if isinstance(v, tuple) else v)
            for k, v in value.items()
        }
    elif isinstance(value, tuple):
        metadata_safe[key] = list(value)
    else:
        metadata_safe[key] = value

json_artifacts['pipeline_metadata.json'] = metadata_safe

for filename, obj in json_artifacts.items():
    filepath = os.path.join(artifact_dir, filename)
    with open(filepath, 'w') as f:
        json.dump(obj, f, indent=2, default=str)

print(f"\n  Artifacts saved to '{artifact_dir}/':")
for f_name in json_artifacts:
    print(f"     {f_name}")
print(f"     scaler.pkl")


  Artifacts saved to 'artifacts/':
     etl_config.json
     feature_names.json
     feature_names_all.json
     imputation_values.json
     outlier_bounds.json
     type_mapping.json
     shift_mapping.json
     pipeline_metadata.json
     scaler.pkl


In [16]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Load artifacts into fresh state variables
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

inf_artifact_dir = 'artifacts'

inf_scaler = joblib.load(os.path.join(inf_artifact_dir, 'scaler.pkl'))

def _load_json(fname):
    fp = os.path.join(inf_artifact_dir, fname)
    if os.path.exists(fp):
        with open(fp, 'r') as f:
            return json.load(f)
    return None

inf_config              = _load_json('etl_config.json')        or config
inf_feature_names       = _load_json('feature_names.json')     or []
inf_imputation_values   = _load_json('imputation_values.json') or {}
inf_outlier_bounds      = _load_json('outlier_bounds.json')    or {}
inf_type_mapping        = _load_json('type_mapping.json')      or type_mapping
inf_shift_mapping       = _load_json('shift_mapping.json')     or shift_mapping

print(f"Artifacts loaded from '{inf_artifact_dir}/'")
print(f"   Features : {len(inf_feature_names)}")
print(f"   Scaler   : {type(inf_scaler).__name__}")

Artifacts loaded from 'artifacts/'
   Features : 18
   Scaler   : RobustScaler


In [17]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Run inference on a small sample
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 60)
print("  RUNNING INFERENCE PIPELINE")
print("=" * 60 + "\n")

# Grab 5 rows, drop target to simulate real incoming data
sample = df_raw.head(5).drop(columns=[target_col], errors='ignore').copy()

# ── Impute ───────────────────────────────
for col, fill_val in inf_imputation_values.items():
    if col in sample.columns:
        sample[col].fillna(fill_val, inplace=True)

# ── Outlier clipping ─────────────────────
for col, bounds in inf_outlier_bounds.items():
    if col in sample.columns:
        sample[col] = sample[col].clip(
            lower=bounds['lower'],
            upper=bounds['upper'],
        )

# ── Feature engineering (identical to training) ──

sample['Power_W']            = sample['Torque_Nm'] * sample['Rot_Speed_RPM'] * (2 * np.pi / 60)
sample['Strain_Level']       = sample['Torque_Nm'] * sample['Tool_Wear_Min']
sample['Torque_Speed_Ratio'] = sample['Torque_Nm'] / (sample['Rot_Speed_RPM'] + 1)
sample['Efficiency_Proxy']   = sample['Power_W'] / (sample['Tool_Wear_Min'] + 1)

sample['Temp_Diff']          = sample['Process_Temp_K'] - sample['Air_Temp_K']
sample['Temp_Ratio']         = sample['Process_Temp_K'] / (sample['Air_Temp_K'] + 1)
sample['Heat_Risk']          = sample['Temp_Diff'] * sample['Rot_Speed_RPM'] / 1000
sample['Overheat_Flag']      = (sample['Process_Temp_K'] > 312).astype(int)

if 'Vibration_mm_s' in sample.columns:
    sample['Vib_Wear_Interaction'] = sample['Vibration_mm_s'] * sample['Tool_Wear_Min']
    sample['Vib_Risk_Level'] = pd.cut(
        sample['Vibration_mm_s'],
        bins=[0, 1.0, 2.0, 3.5, float('inf')],
        labels=[0, 1, 2, 3],
    ).astype(float).fillna(0).astype(int)
    sample['Vib_Torque'] = sample['Vibration_mm_s'] * sample['Torque_Nm']

if 'Pressure_bar' in sample.columns:
    sample['Pressure_Deviation'] = abs(sample['Pressure_bar'] - 6.0)
    sample['Pressure_Torque']    = sample['Pressure_bar'] * sample['Torque_Nm']
    sample['Low_Pressure_Flag']  = (sample['Pressure_bar'] < 4.5).astype(int)

sample['Wear_Stage'] = pd.cut(
    sample['Tool_Wear_Min'],
    bins=[-1, 50, 120, 200, float('inf')],
    labels=[0, 1, 2, 3],
).astype(float).fillna(0).astype(int)

sample['Load_Regime'] = pd.cut(
    sample['Power_W'],
    bins=[-float('inf'), 3000, 6000, 9000, float('inf')],
    labels=[0, 1, 2, 3],
).astype(float).fillna(0).astype(int)

sample['Wear_Rate'] = sample['Tool_Wear_Min'] / (sample['Torque_Nm'] + 1)

if 'Machine_Age_Years' in sample.columns:
    sample['Age_Stress']    = sample['Machine_Age_Years'] * sample['Strain_Level'] / 100
    sample['Age_Vibration'] = sample['Machine_Age_Years'] * sample.get('Vibration_mm_s', 0)

for col in ['Rot_Speed_RPM', 'Torque_Nm', 'Tool_Wear_Min']:
    col_mean = sample[col].mean()
    col_std  = sample[col].std()
    if col_std and col_std > 0:
        sample[f'{col}_zscore'] = abs((sample[col] - col_mean) / col_std)

if 'Vibration_mm_s' in sample.columns:
    col_mean = sample['Vibration_mm_s'].mean()
    col_std  = sample['Vibration_mm_s'].std()
    if col_std and col_std > 0:
        sample['Vibration_mm_s_zscore'] = abs((sample['Vibration_mm_s'] - col_mean) / col_std)

zs_feats = [c for c in sample.columns if '_zscore' in c]
if zs_feats:
    sample['Anomaly_Score'] = sample[zs_feats].mean(axis=1)

if 'Timestamp' in sample.columns:
    sample['Timestamp'] = pd.to_datetime(sample['Timestamp'])
    sample['Hour']      = sample['Timestamp'].dt.hour
    sample['Hour_Sin']  = np.sin(2 * np.pi * sample['Hour'] / 24)
    sample['Hour_Cos']  = np.cos(2 * np.pi * sample['Hour'] / 24)
    sample['DayOfWeek'] = sample['Timestamp'].dt.dayofweek
    sample['Is_Weekend'] = (sample['DayOfWeek'] >= 5).astype(int)

if 'Type' in sample.columns:
    sample['Type_Encoded'] = sample['Type'].map(inf_type_mapping).fillna(0).astype(int)
if 'Shift' in sample.columns:
    sample['Shift_Encoded'] = sample['Shift'].map(inf_shift_mapping).fillna(0).astype(int)

# ── Build matrix ─────────────────────────
inf_exclude = set(
    id_cols + categorical_cols + failure_mode_cols +
    [target_col, 'Machine_Age_Years', 'Hour']
)
inf_numeric = sample.select_dtypes(include=[np.number]).columns.tolist()
inf_feat_cols = [c for c in inf_numeric if c not in inf_exclude]

X_inf = sample[inf_feat_cols].fillna(0).replace([np.inf, -np.inf], 0)

# ── Scale ────────────────────────────────
X_inf_scaled = pd.DataFrame(
    inf_scaler.transform(X_inf),
    columns=X_inf.columns,
    index=X_inf.index,
)

# ── Select saved features ────────────────
available = [f for f in inf_feature_names if f in X_inf_scaled.columns]
missing   = [f for f in inf_feature_names if f not in X_inf_scaled.columns]
if missing:
    logger.warning(f"  Missing features (filling with 0): {missing}")
    for col in missing:
        X_inf_scaled[col] = 0

X_inference = X_inf_scaled[inf_feature_names]

print(f"  Inference output shape: {X_inference.shape}")
print(f"\n  Inference test passed — output shape: {X_inference.shape}")
X_inference


  RUNNING INFERENCE PIPELINE

  Inference output shape: (5, 18)

  Inference test passed — output shape: (5, 18)


,Vib_Torque,Strain_Level,Tool_Wear_Min,Vibration_mm_s,Tool_Wear_Min_zscore,Vib_Risk_Level,Age_Vibration,Wear_Rate,Age_Stress,Efficiency_Proxy,Torque_Nm,Vibration_mm_s_zscore,Temp_Diff,Type_Encoded,Heat_Risk,Shift_Encoded,DayOfWeek,Process_Temp_K
0,-0.608899,-0.816333,-0.847764,-0.866099,-0.501451,-1.0,-0.855437,-0.835478,-0.637018,19.869316,1.047129,0.245251,-0.213630,0.0,0.566069,0.5,-1.0,0.187341
1,-0.895857,-0.799517,-0.831589,-1.063351,0.755640,-1.0,-0.483376,-0.820519,-0.558860,11.648754,0.724936,0.148910,-0.179449,2.0,-0.281753,0.5,-1.0,0.078729
2,-0.925006,-0.841728,-0.862036,-0.999331,-0.309190,-1.0,-0.375064,-0.841607,-0.617084,19.779205,-0.041988,-0.564011,-0.153814,0.0,-0.345285,0.5,-1.0,-0.503553
3,-0.754409,-0.844063,-0.870599,-1.064216,0.356329,-1.0,-0.581543,-0.853862,-0.627070,45.954264,1.940874,0.158544,0.444350,2.0,0.153704,0.5,-1.0,0.796776
4,-0.943758,-0.842669,-0.856327,-0.852257,-0.752870,-1.0,-0.637580,-0.827310,-0.634547,12.095922,-0.934019,0.399396,-0.452896,1.0,-0.428182,0.5,-1.0,-0.826372


In [18]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Save processed train/test data for the Training notebook
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

os.makedirs('artifacts', exist_ok=True)

X_train_balanced.to_csv('artifacts/X_train.csv', index=False)
X_test_scaled.to_csv('artifacts/X_test.csv', index=False)

y_train_balanced.to_frame('Machine_Failure').to_csv('artifacts/y_train.csv', index=False)
y_test.to_frame('Machine_Failure').to_csv('artifacts/y_test.csv', index=False)

# Also save the unbalanced versions
X_train_unbalanced.to_csv('artifacts/X_train_unbalanced.csv', index=False)
y_train_unbalanced.to_frame('Machine_Failure').to_csv('artifacts/y_train_unbalanced.csv', index=False)

# EDA data sample
sample_size = min(10000, len(data))
data.sample(n=sample_size, random_state=42).to_csv('artifacts/eda_data.csv', index=False)

print(" Processed train/test data saved to artifacts/")
print(f"   X_train : {X_train_balanced.shape}")
print(f"   X_test  : {X_test_scaled.shape}")
print(f"   y_train : {y_train_balanced.shape}")
print(f"   y_test  : {y_test.shape}")

 Processed train/test data saved to artifacts/
   X_train : (58047, 18)
   X_test  : (10000, 18)
   y_train : (58047,)
   y_test  : (10000,)
